# NB01: Baselines and EDA

**Outputs**
- `data/baseline_results.csv` — B0–B5 RMSE per target and fold
- EDA figures in `figures/`

**Run on**: JupyterHub (NB00 must be run first)

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

for _cand in [Path.cwd() / 'scripts', Path.cwd().parent / 'scripts']:
    if _cand.exists():
        sys.path.insert(0, str(_cand))
        break

DATA_DIR = next(p for p in [Path.cwd() / 'data', Path.cwd().parent / 'data'] if p.exists())
FIGURES_DIR = next(p for p in [Path.cwd() / 'figures', Path.cwd().parent / 'figures'] if p.exists())
FIGURES_DIR.mkdir(exist_ok=True)

feature_matrix = pd.read_parquet(DATA_DIR / 'feature_matrix.parquet')
spatial_blocks = pd.read_csv(DATA_DIR / 'spatial_blocks.csv').set_index('sample_id')
block_labels = spatial_blocks['block'].values

print(f'Feature matrix: {feature_matrix.shape}')
print(f'Columns: {list(feature_matrix.columns)}')

## 1. EDA — target distributions

In [ ]:
targets = ['log_Cu_ppm', 'log_Zn_ppm', 'log_Pb_ppm', 'log_Ni_ppm']
targets_avail = [t for t in targets if t in feature_matrix.columns]

fig, axes = plt.subplots(1, len(targets_avail), figsize=(4 * len(targets_avail), 4))
for ax, col in zip(axes, targets_avail):
    vals = feature_matrix[col].dropna()
    ax.hist(vals, bins=50, edgecolor='none')
    ax.set_title(col.replace('log_', ''))
    ax.set_xlabel('log1p(ppm)')
plt.suptitle('Metal target distributions (log-transformed)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_target_distributions.png', dpi=150)
plt.show()

In [ ]:
# Target summary stats
print(feature_matrix[targets_avail].describe().round(3))

## 2. EDA — covariate distributions and correlations

In [ ]:
from modelling import ENV_FEATURES, _get_cwm_cols

env_cols = [c for c in ENV_FEATURES if c in feature_matrix.columns]
cwm_cols = _get_cwm_cols(feature_matrix)
print(f'Env features present ({len(env_cols)}): {env_cols}')
print(f'CWM features ({len(cwm_cols)}): {cwm_cols}')

# Correlation matrix: env + CWM vs targets
feature_cols = env_cols + cwm_cols
corr = feature_matrix[feature_cols + targets_avail].corr(method='spearman')
corr_targets = corr.loc[feature_cols, targets_avail]

fig, ax = plt.subplots(figsize=(max(6, len(targets_avail) * 1.5), len(feature_cols) * 0.4 + 1))
im = ax.imshow(corr_targets.T.values, cmap='RdBu_r', vmin=-0.5, vmax=0.5, aspect='auto')
ax.set_xticks(range(len(feature_cols)))
ax.set_xticklabels(feature_cols, rotation=90, fontsize=8)
ax.set_yticks(range(len(targets_avail)))
ax.set_yticklabels([t.replace('log_', '') for t in targets_avail])
plt.colorbar(im, ax=ax, label='Spearman r')
plt.title('Feature–target Spearman correlations')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_feature_target_correlations.png', dpi=150)
plt.close()

## 3. Spatial map

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sc = ax.scatter(
    feature_matrix['lon'], feature_matrix['lat'],
    c=block_labels, cmap='tab10', s=5, alpha=0.5
)
plt.colorbar(sc, ax=ax, label='Block')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Spatial blocks (k=5 geographic clusters)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_spatial_blocks.png', dpi=150)
plt.show()

## 4. Baselines B0–B5

In [ ]:
from modelling import build_ridge, build_xgboost, rmse, _drop_nan_rows, ENV_FEATURES
from spatial_utils import spatial_cv_splits

cwm_cols = [c for c in feature_matrix.columns if c.startswith('CWM_')]
env_cols = [c for c in ENV_FEATURES if c in feature_matrix.columns]
splits = list(spatial_cv_splits(block_labels))

targets_avail = [t for t in ['log_Cu_ppm', 'log_Zn_ppm', 'log_Pb_ppm', 'log_Ni_ppm']
                 if t in feature_matrix.columns]

baseline_records = []

for target in targets_avail:
    y = feature_matrix[target]

    # B0: intercept-only
    oof = np.full(len(y), np.nan)
    for train_idx, test_idx in splits:
        oof[test_idx] = y.iloc[train_idx].mean()
    valid = ~np.isnan(oof) & y.notna()
    baseline_records.append({'model': 'B0', 'target': target, 'rmse': rmse(y.values[valid], oof[valid])})

    # B1: pH only (ridge)
    X_ph = feature_matrix[['ph']]
    oof = np.full(len(y), np.nan)
    for train_idx, test_idx in splits:
        Xtr, ytr = _drop_nan_rows(X_ph.iloc[train_idx], y.iloc[train_idx])
        Xte, yte = _drop_nan_rows(X_ph.iloc[test_idx], y.iloc[test_idx])
        if len(Xtr) > 5 and len(Xte) > 0:
            m = build_ridge().fit(Xtr, ytr)
            oof[test_idx[:len(Xte)]] = m.predict(Xte)
    valid = ~np.isnan(oof) & y.notna()
    baseline_records.append({'model': 'B1', 'target': target, 'rmse': rmse(y.values[valid], oof[valid])})

    # B2: all cheap env (ridge)
    X_env = feature_matrix[env_cols]
    oof = np.full(len(y), np.nan)
    for train_idx, test_idx in splits:
        Xtr, ytr = _drop_nan_rows(X_env.iloc[train_idx], y.iloc[train_idx])
        Xte, yte = _drop_nan_rows(X_env.iloc[test_idx], y.iloc[test_idx])
        if len(Xtr) > 5 and len(Xte) > 0:
            m = build_ridge().fit(Xtr, ytr)
            oof[test_idx[:len(Xte)]] = m.predict(Xte)
    valid = ~np.isnan(oof) & y.notna()
    baseline_records.append({'model': 'B2', 'target': target, 'rmse': rmse(y.values[valid], oof[valid])})

    # B3: lat/lon only (ridge)
    X_latlon = feature_matrix[['lat', 'lon']].fillna(feature_matrix[['lat', 'lon']].median())
    oof = np.full(len(y), np.nan)
    for train_idx, test_idx in splits:
        Xtr, ytr = _drop_nan_rows(X_latlon.iloc[train_idx], y.iloc[train_idx])
        Xte, yte = _drop_nan_rows(X_latlon.iloc[test_idx], y.iloc[test_idx])
        if len(Xtr) > 5 and len(Xte) > 0:
            m = build_ridge().fit(Xtr, ytr)
            oof[test_idx[:len(Xte)]] = m.predict(Xte)
    valid = ~np.isnan(oof) & y.notna()
    baseline_records.append({'model': 'B3', 'target': target, 'rmse': rmse(y.values[valid], oof[valid])})

    # B4: pH only XGBoost
    oof = np.full(len(y), np.nan)
    for train_idx, test_idx in splits:
        Xtr, ytr = _drop_nan_rows(X_ph.iloc[train_idx], y.iloc[train_idx])
        Xte, yte = _drop_nan_rows(X_ph.iloc[test_idx], y.iloc[test_idx])
        if len(Xtr) > 5 and len(Xte) > 0:
            m = build_xgboost(n_estimators=200)
            m.fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=False)
            oof[test_idx[:len(Xte)]] = m.predict(Xte)
    valid = ~np.isnan(oof) & y.notna()
    baseline_records.append({'model': 'B4', 'target': target, 'rmse': rmse(y.values[valid], oof[valid])})

    print(f'Done baselines: {target}')

baseline_df = pd.DataFrame(baseline_records)
print(baseline_df.pivot(index='model', columns='target', values='rmse').round(4))

In [ ]:
baseline_df.to_csv(DATA_DIR / 'baseline_results.csv', index=False)
print('=== NB01 COMPLETE ===')
print(baseline_df.pivot(index='model', columns='target', values='rmse').round(4))